# Reproduction code — Post-pandemic dynamics of influenza B (Victoria & Yamagata)

This notebook reproduces the modelling results and figures of the accompanying
paper. It uses the deterministic + stochastic, two-lineage, three-region
influenza B model implemented in `simulator.py`.

## Figure map
| Section | Output file(s) | Paper figure |
|---|---|---|
| 1. FluNet surveillance data | `nh_data.pdf`, `data.pdf` | Fig. 1, Fig. S1 |
| 2. Latin Hypercube search | `search_1M.csv` | input to all analyses |
| 3. Parameter-space analysis | `shap_beeswarm.png`, `both_outcomes*.png`, `evaluation_densities.png` | Table 1, Figs. 2/S2, S3, 3 |
| 4. Likelihood scoring & NH best fit | `sto_nh.png` | Fig. 4 |
| 5. Reintroduction analysis | `Yam.png`, `Vic_scatter.png`, `Yam_scatter.png`, `Yam_dif.png` | Fig. 5 (+ SI scatters) |

## Data files (must sit in the same directory as this notebook)
- `FluNet_NH.csv`, `FluNet_Tr.csv`, `FluNet_SH.csv` — WHO FluNet weekly influenza B reports.

## Reproducibility notes
- `search_1M.csv` is **not** included in this deposit (it is ~200 MB). Section 2 generates it
  from scratch; the search is deterministic (`seed=42`) but takes several hours on 8 cores.
  Everything from Section 3 onwards depends on it, so Section 2 must be run first. Once written,
  the file is reused on subsequent runs and the search is skipped.
- Some Section 4/5 cells run the model many thousands of times (large stochastic ensembles
  and 559-sample loops) and can be slow — see the inline notes.
- See `README.md` for environment setup (`requirements.txt`) and instructions.


## 1. FluNet surveillance data (Fig. 1 and Fig. S1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

df = pd.read_csv('FluNet_NH.csv')

df['ISO_WEEKSTARTDATE'] = pd.to_datetime(df['Week start date (ISO 8601 calendar)'])

start_date = '2015-10-01'
end_date = '2025-09-30'

BYAM = 'B (Yamagata)'
BVIC = 'B (Victoria)'
BNOTDETERMINED = 'B (lineage not determined)'

cols_to_fill = [BYAM, BVIC, BNOTDETERMINED]

filtered_df = df[
        (df['ISO_WEEKSTARTDATE'] >= start_date) &
        (df['ISO_WEEKSTARTDATE'] < end_date)
    ].copy()
    
for col in cols_to_fill:
    filtered_df[col] = filtered_df[col].fillna(0)

# Group by week to aggregate counts
filtered_weekly = filtered_df.groupby('ISO_WEEKSTARTDATE')[cols_to_fill].sum().reset_index()

# Calculate total B
filtered_weekly['Total_B'] = filtered_weekly[BYAM] + filtered_weekly[BVIC]

# Filter out weeks with 0 cases
plot_data = filtered_weekly[filtered_weekly['Total_B'] > 0].copy()
    

# Pandemic cutoff date
pandemic_cutoff = pd.to_datetime('2020-04-01')
testing_cutoff = pd.to_datetime('2020-07-01')

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(plot_data['ISO_WEEKSTARTDATE'], np.sqrt(plot_data[BVIC]),  label='Victoria')
ax.plot(plot_data['ISO_WEEKSTARTDATE'], np.sqrt(plot_data[BYAM]),  label='Yamagata')
ax.plot(plot_data['ISO_WEEKSTARTDATE'], np.sqrt(plot_data[BNOTDETERMINED]), label='Untyped')
ax.axvline(pandemic_cutoff, color='red', linestyle='--', linewidth=2, alpha=0.7)

lim = np.sqrt(1.05 * max(plot_data[[BVIC, BYAM, BNOTDETERMINED]].max().max(), 1))
ax.text(pandemic_cutoff, lim*0.95, 'lockdown', color='red', fontsize=14,  alpha=0.7, va='top', ha='left', rotation=90)

ax.set_ylim(0, lim)
ax.set_ylabel('weekly reports', fontsize=14)

ax.legend(loc='upper left', fontsize=14)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax.xaxis.get_majorticklabels(), fontsize=12)
plt.setp(ax.yaxis.get_majorticklabels(), fontsize=12)
# Set x-axis limits
ax.set_xlim(pd.Timestamp(start_date), pd.Timestamp(end_date))

plt.tight_layout()
plt.savefig('nh_data.pdf')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# Load the data for all three regions
df_nh = pd.read_csv('FluNet_NH.csv')
df_tr = pd.read_csv('FluNet_Tr.csv')
df_sh = pd.read_csv('FluNet_SH.csv')

dfs = [df_nh, df_tr, df_sh]
# Convert date column to datetime objects
for df in dfs:
    df['ISO_WEEKSTARTDATE'] = pd.to_datetime(df['Week start date (ISO 8601 calendar)'])

start_date ='2015-10-01'
end_date = '2025-09-30'

BYAM = 'B (Yamagata)'
BVIC = 'B (Victoria)'
BNOTDETERMINED = 'B (lineage not determined)'

cols_to_fill = [BYAM, BVIC, BNOTDETERMINED]
region_names = ["Northern Hemisphere", "Tropics", "Southern Hemisphere"]
region_dfs = []

# Process each region
for region_df in dfs:
    filtered_df = region_df[
        (region_df['ISO_WEEKSTARTDATE'] >= start_date) &
        (region_df['ISO_WEEKSTARTDATE'] < end_date)
    ].copy()
    
    for col in cols_to_fill:
        filtered_df[col] = filtered_df[col].fillna(0)
    
    # Group by week to aggregate counts
    filtered_weekly = filtered_df.groupby('ISO_WEEKSTARTDATE')[cols_to_fill].sum().reset_index()
    
    # Calculate total B
    filtered_weekly['Total_B'] = filtered_weekly[BYAM] + filtered_weekly[BVIC]
    
    # Filter out weeks with 0 cases
    plot_data = filtered_weekly[filtered_weekly['Total_B'] > 0].copy()
    
    region_dfs.append(plot_data)
    

# Pandemic cutoff date
pandemic_cutoff = pd.to_datetime('2020-04-01')
testing_cutoff = pd.to_datetime('2020-07-01')

# Create subplots for all three regions
fig, axes = plt.subplots(len(dfs), 1, figsize=(15, 12), sharex=True)

for idx, (ax, region_data) in enumerate(zip(axes, region_dfs)):
    ax.plot(region_data['ISO_WEEKSTARTDATE'], np.sqrt(region_data[BVIC]), label='Victoria')
    ax.plot(region_data['ISO_WEEKSTARTDATE'], np.sqrt(region_data[BYAM]), label='Yamagata')
    ax.plot(region_data['ISO_WEEKSTARTDATE'], np.sqrt(region_data[BNOTDETERMINED]), label='Untyped')
    lim=np.sqrt(1.05*max(region_data[BVIC].max(), region_data[BYAM].max(), region_data[BNOTDETERMINED].max()))
    ax.vlines(pandemic_cutoff, 0, lim, 
            colors='red', linestyles='--', linewidth=2, alpha=0.7)
    ax.text(pandemic_cutoff, lim*0.95, 'lockdown', color='red', fontsize=14,  alpha=0.7, va='top', ha='left', rotation=90)
    ax.set_ylim(0, lim)
    ax.set_ylabel('weekly reports', fontsize=14)
    ax.set_title(f'{region_names[idx]}', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
    # Format x-axis for bottom subplot
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


# Set x-axis limits
for ax in axes:
    ax.set_xlim(pd.Timestamp(start_date), pd.Timestamp(end_date))
    plt.setp(ax.yaxis.get_majorticklabels(), fontsize=12)
#axes[-1].set_xlabel('ISO Week Start Date', fontsize=11)
plt.setp(axes[-1].xaxis.get_majorticklabels(), fontsize=12)

plt.tight_layout()
plt.savefig('data.pdf')
plt.show()

## 2. Latin Hypercube parameter search (produces `search_1M.csv`)

The cell below defines the parallel Latin Hypercube sampling (LHS) search over the
ten model parameters. `search_1M.csv` is not bundled with this deposit, so the search
must be run once to generate it — every later section reads that file. The search is
**deterministic** (`seed=42`) but expensive (~hours on 8 cores); once the file exists it
is reused and the search is skipped.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import qmc
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm # tqdm.auto formats beautifully in Jupyter

# 1. Import the wrapper from your file! 
from simulator import run_single_simulation
#WITH THRESHOLD one FOR EXTINCTION

# 2. Define your bounds
param_names = ['R0V', 'R0Y', 'TV', 'TY', 'sigmaVY', 'sigmaYV', 'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']
lower_bounds = [1, 1, 1, 1, 0.0, 0.0, 0.0010,0,0.2,100]
upper_bounds = [5, 5, 10,  10,  1.0, 1.0, 0.01, 100, 0.8,600] 

def run_parallel_lhs_jupyter(n_samples=2000, n_cores=8):
    print(f"Setting up Latin Hypercube with {n_samples} samples...")
    sampler = qmc.LatinHypercube(d=len(param_names), seed=42)
    sample_scaled = qmc.scale(sampler.random(n=n_samples), lower_bounds, upper_bounds)
    
    tasks = [row for row in sample_scaled]
    results = []
    failed =[]
    
   
    
    if n_cores == 1:
        print("Running simulations sequentially...")
        for task in tqdm(tasks, total=n_samples):
            res = run_single_simulation(task)
            if res['success']:
                results.append(res)
            else:
                failed.append(res)
    else:
        # ProcessPoolExecutor handles the core management
        with ProcessPoolExecutor(max_workers=n_cores) as executor:
            print(f"Firing up {n_cores} cores...")
            # Map the imported wrapper function to your LHS tasks
            #for res in tqdm(executor.map(run_single_simulation, tasks), total=n_samples)
            for res in tqdm(executor.map(run_single_simulation, sample_scaled, chunksize=100), total=n_samples):
                (results if res['success'] else failed).append(res)
                
    # Clean up and filter
    df = pd.DataFrame(results).drop(columns=['success'])
    
    print("-" * 50)
    print(f"Total successful ODE runs: {len(df)} / {n_samples}")
    print("-" * 50)
    
    return df, failed

In [ ]:
# Runs the one-million-sample search and writes `search_1M.csv`, which every later section
# reads. Deterministic (seed=42) but expensive (~hours on 8 cores). The file is not bundled
# with this deposit, so this must run once; delete the file to force a fresh rerun.
from pathlib import Path

RUN_SEARCH = not Path('search_1M.csv').exists()

if RUN_SEARCH:
    all_res, failed = run_parallel_lhs_jupyter(n_samples=1_000_000, n_cores=8)
    all_res.to_csv('search_1M.csv', index=False)
    print(f"Wrote search_1M.csv with {len(all_res):,} successful runs.")
else:
    print('search_1M.csv already present — skipping the search.')


In [ ]:
import pandas as pd
all_res = pd.read_csv('search_1M.csv')
print(f"Loaded {len(all_res):,} parameter samples from search_1M.csv")


## 3. Parameter-space analysis (Table 1, Figs. 2/S2, S3, 3)

XGBoost + SHAP feature importance (Fig. S3 and the mean-|SHAP| column of Table 1),
the parameter-distribution plots (Fig. 2/S2), and the 2-D parameter-space projection
(Fig. 3, `evaluation_densities.png`).

In [ ]:
import xgboost as xgb
import shap
import numpy as np
import matplotlib.pyplot as plt

# Parameters
param_cols = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'sigmaVY', 'sigmaYV',
              'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']



NEG_TO_POS_RATIO = 10   # keep this many negatives per positive after undersampling
RANDOM_STATE     = 42

# Binary outcome (same definition as cond_green used elsewhere)
_df = all_res.copy()


coexistence_target = (_df['simple_coexist'] == True) & (_df['recovery_Y'] != -1) & (_df['recovery_V'] != -1)
Yextinction_Vrecovery = coexistence_target & (_df['extinct_Y'] == True) & (_df['extinct_V'] == False) & (_df['recovery_Y'] != -100)

y_all = (Yextinction_Vrecovery & (_df['recovery_V'] > 1.5) & (_df['recovery_V'] <= 2)).astype(int)

X_all = _df[param_cols].reset_index(drop=True)
y_all = y_all.reset_index(drop=True)

n_pos = int(y_all.sum())
n_neg = int((y_all == 0).sum())
print(f"Class balance: {n_pos:,} pos / {n_neg:,} neg  ({n_pos/(n_pos+n_neg)*100:.3f}% positive)")

# Stratified undersample 
pos_idx  = y_all[y_all == 1].index
neg_idx  = y_all[y_all == 0].index
n_keep   = min(len(neg_idx), len(pos_idx) * NEG_TO_POS_RATIO)
neg_keep = neg_idx.to_series().sample(n=n_keep, random_state=RANDOM_STATE).index
keep     = pos_idx.union(neg_keep)

X_bal = X_all.loc[keep]
y_bal = y_all.loc[keep]
print(f"Downsampled: {int(y_bal.sum()):,} pos / {int((y_bal==0).sum()):,} neg  ({NEG_TO_POS_RATIO}:1 ratio)")

# XGBoost 
clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
clf.fit(X_bal, y_bal)
print("XGBoost training done.")

#### SHAP
explainer = shap.TreeExplainer(clf)
shap_raw  = explainer.shap_values(X_bal)
shap_vals = shap_raw[1] if isinstance(shap_raw, list) else shap_raw   # positive class



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=param_cols, name="Mean |SHAP|")
display_labels = {
    'R0V':              r"$R_{0,V}$",
    'R0Y':              r"$R_{0,Y}$",
    'TV_years':         r"$T_V$",
    'TY_years':         r"$T_Y$",
    'sigmaVY':          r"$\sigma_{VY}$",
    'sigmaYV':          r"$\sigma_{YV}$",
    'm':                r"$m$",
    'season_phase':     r"$\phi$",
    'c_pandemic':       r"$c_0$",
    'Lockdown_duration': r"$\tau$",
}
X_bal_labeled = X_bal.rename(columns=display_labels)

# mapping: display label -> mean |SHAP|
label_to_shap = {display_labels.get(col, col): mean_abs_shap[i]
                 for i, col in enumerate(param_cols)}

# -- Plot 1: beeswarm (direction + spread) ------------------------------------
shap.summary_plot(shap_vals, X_bal_labeled, max_display=len(param_cols), show=False)
ax = plt.gca()
ax.set_yticklabels([
    f"{lbl.get_text()}  ({label_to_shap[lbl.get_text()]:.1f})"
    if lbl.get_text() in label_to_shap else lbl.get_text()
    for lbl in ax.get_yticklabels()
])
ax.text(0.135, 0.95, "Feature (mean |SHAP|)",
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=12, fontweight='bold', clip_on=False)
#plt.title('SHAP Beeswarm', fontsize=13, weight='bold', pad=10)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

# -- Plot 2: mean |SHAP| bar chart --------------------------------------------
"""
shap_sorted = shap_importance.sort_values()
bar_labels  = [display_labels.get(c, c) for c in shap_sorted.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(bar_labels, shap_sorted.values, color='steelblue', edgecolor='black', alpha=0.85)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12, weight='bold')
ax.set_title('Feature Importance -- XGBoost + SHAP', fontsize=13, weight='bold', pad=10)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
"""

print("\nMean |SHAP| (descending):")
print(shap_importance.sort_values(ascending=False).to_string())

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_parameter_space(df, name=None):    
    """
    Takes the full DataFrame from the LHS sweep and plots the parameter spaces.
    Duplicates the 'brown' subset rows so they appear in both the full 'green' 
    distribution and their own 'brown' distribution.
    """
    plot_df = df.copy()
    
    coexistence_target = (df['simple_coexist'] == True) & (df['recovery_Y'] !=-1) & (df['recovery_V'] != -1)
    Yextinction_Vrecovery = coexistence_target & (df['extinct_Y'] == True) & (df['extinct_V'] == False) & (df['recovery_Y'] != -100)
    
    str1 = 'Vic and Yam coexist before COVID-19'
    str3 = 'Yam extinct, Vic reemerges Oct 2021 - Apr 2022 (All)'
    str_blue = 'Yam extinct, Vic reemerges Oct 2021 - Apr 2022 (Favored Subset)' 

    # ==========================================
    # DEFINING OUTCOMES (Overlapping Logic)
    # ==========================================

    # 1. Define the base Green condition
    cond_green = Yextinction_Vrecovery & (df['recovery_V'] > 1.5) & (df['recovery_V'] <= 2) 

    # 2. Assign the baseline and the FULL green group (this includes the brown ones for now)
    plot_df['Outcome'] = np.select([cond_green], [str3], default=str1)
    
    # 3. Define the B subset condition
    cond_blue = cond_green & (df['sigmaVY'] > df['sigmaYV'])  & (df['TV_years'] > df['TY_years']) #& (df['R0V'] > df['R0Y'])
    
    # 4. DUPLICATE the blue rows so they appear in both categories
    brown_df = plot_df[cond_blue].copy()
    brown_df['Outcome'] = str_blue # Relabel the duplicates
    
    # Concatenate them back together
    plot_df = pd.concat([plot_df, brown_df], ignore_index=True)
    
    # Define the exact columns from the dataframe
    param_cols = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'sigmaVY', 'sigmaYV', 'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']

    # Create a dictionary mapping the columns to their display names
    display_labels = {'R0V': "B/Vic's basic reproduction number: " + r"$R_{0,V}$",
        'R0Y': "B/Yam's basic reproduction number: " + r"$R_{0,Y}$",
        'TV_years': "Average duration of B/Vic's immunity (years): " + r"$T_{V}$",
        'TY_years': "Average duration of B/Yam's immunity (years): " + r"$T_{Y}$",
        'sigmaVY': "Cross-immunity from B/Vic to B/Yam: " + r"$\sigma_{VY}$",
        'sigmaYV': "Cross-immunity from B/Yam to B/Vic: " + r"$\sigma_{YV}$",
        'm': "Inter-region contact rate (relative): " + r"$m$",
        'season_phase': "Peak seasonality in NH (days since Jan 1st): " + r"$\phi$",
        'c_pandemic': "Initial COVID-19 reduction in contacts (relative): "+r"$c_0$",
        'Lockdown_duration': 'Lockdown timescale (days): ' + r"$\tau$",
    }

    # ==========================================
    # DEFINING COLORS
    # ==========================================
    colors = ['lightgrey', 'green', 'blue']
    palette = {
        str1: colors[0],       
        str3: colors[1],       
        str_blue: colors[2]   
    }
    
    # ==========================================
    # PLOT: Violin Plots
    # ==========================================
    print("Generating Violin Plots...")

    min_max_dict = {}

    # Normalize the data 
    norm_df = plot_df.copy()
    for col in param_cols:
        min_val = norm_df[col].min()
        max_val = norm_df[col].max()
        min_max_dict[col] = (min_val, max_val)
        if max_val > min_val: 
            norm_df[col] = (norm_df[col] - min_val) / (max_val - min_val)

    # Melt the dataframe
    melted_df = norm_df.melt(
        id_vars=['Outcome'], 
        value_vars=param_cols, 
        var_name='Parameter', 
        value_name='Normalized Value'
    )

    # Swap the original column names for the pretty labels
    melted_df['Parameter'] = melted_df['Parameter'].map(display_labels).fillna(melted_df['Parameter'])

    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Explicitly define the hue order to calculate text spacing offsets reliably
    hue_order_list = [str1, str3, str_blue]

    sns.violinplot(
        data=melted_df, 
        y='Parameter', 
        x='Normalized Value', 
        hue='Outcome', 
        hue_order=hue_order_list,
        palette=palette,
        inner='quartile',   
        linewidth=1,
        cut=0,              
        ax=ax
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

    ax.set_xticks([])
    ax.set_xlabel('')
    ax.set_ylabel('') 
    
    ax.tick_params(axis='y', pad=55, length=0, labelsize=12)
    
    # Define vertical dodge offsets for 3 categories in Seaborn
    hue_offsets = [-0.267, 0, 0.267]
    
    for i, col in enumerate(param_cols):
        min_val, max_val = min_max_dict[col]
        
        # Custom display formatting logic per column rules
        if col == 'Lockdown_duration' or col =='season_phase':
            fmt = lambda x: f"{x:.0f}"
        elif col == 'm':
            fmt = lambda x: f"{x:.4f}"
        else:
            fmt = lambda x: f"{x:.2f}"
            
        ax.text(-0.02, i, fmt(min_val), va='center', ha='right', fontsize=12, color='black')
        ax.text(1.02, i, fmt(max_val), va='center', ha='left', fontsize=12, color='black')
        
        # Inject raw percentile numbers inside the violins with matching rules
        for j, outcome in enumerate(hue_order_list):
            data_raw = plot_df[plot_df['Outcome'] == outcome][col].dropna().values
            if len(data_raw) < 2:
                continue
                
            q25, q50, q75 = np.percentile(data_raw, [25, 50, 75])
            mean = np.mean(data_raw)
            # Normalize internal positioning matrix matching visual axis mapping scales
            q25_n = (q25 - min_val) / (max_val - min_val) if max_val > min_val else 0.5
            q50_n = (q50 - min_val) / (max_val - min_val) if max_val > min_val else 0.5
            q75_n = (q75 - min_val) / (max_val - min_val) if max_val > min_val else 0.5
            
            y_pos = i + hue_offsets[j]
            bbox_style = dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.8)
            
            str_me=fmt(q50)+' ('+fmt(mean)+')'
            # Write out formatted annotations
            ax.text(q25_n, y_pos, fmt(q25), ha='right', va='center', fontsize=8, bbox=bbox_style, zorder=10)
            ax.text(q50_n, y_pos, str_me, ha='center', va='center', fontsize=8, weight='bold', bbox=bbox_style, zorder=10)
            ax.text(q75_n, y_pos, fmt(q75), ha='left', va='center', fontsize=8, bbox=bbox_style, zorder=10)

    sns.despine(ax=ax, left=True, bottom=True)
    plt.tight_layout()
    
    if name is not None:
        plt.savefig(name)
    plt.show()

    # ==========================================
    # PLOT 2: 2D Parameter Subplots for Green Group (KDE Densities)
    # ==========================================
    print("Generating Green Subset Subplots...")
    
    green_df = df[cond_green].copy()
    fig2, axes = plt.subplots(1, 4, figsize=(24, 6))
    kde_kws = {'fill': True, 'cmap': 'Greens', 'thresh': 0.05, 'levels': 15}
    fontsize = 14
    
    sns.kdeplot(data=green_df, x='R0Y', y='TY_years', ax=axes[0], **kde_kws)
    axes[0].set_xlabel(display_labels['R0Y'], fontsize=fontsize)
    axes[0].set_xlim(min_max_dict['R0Y'])
    axes[0].set_ylabel(display_labels['TY_years'], fontsize=fontsize)
    axes[0].set_ylim(min_max_dict['TY_years'])
    
    sns.kdeplot(data=green_df, x='R0V', y='TV_years', ax=axes[1], **kde_kws)
    axes[1].set_xlabel(display_labels['R0V'], fontsize=fontsize)
    axes[1].set_xlim(min_max_dict['R0V'])
    axes[1].set_ylabel(display_labels['TV_years'], fontsize=fontsize)
    axes[1].set_ylim(min_max_dict['TV_years'])
    
    sns.kdeplot(data=green_df, x='R0V', y='Lockdown_duration', ax=axes[2], **kde_kws)
    axes[2].set_xlabel(display_labels['R0V'], fontsize=fontsize)
    axes[2].set_xlim(min_max_dict['R0V'])
    axes[2].set_ylabel(display_labels['Lockdown_duration'], fontsize=fontsize)
    axes[2].set_ylim(min_max_dict['Lockdown_duration'])
    
    sns.kdeplot(data=green_df, x='c_pandemic', y='Lockdown_duration', ax=axes[3], **kde_kws)
    axes[3].set_xlabel(display_labels['c_pandemic'], fontsize=fontsize)
    axes[3].set_xlim(min_max_dict['c_pandemic'])
    axes[3].set_ylabel(display_labels['Lockdown_duration'], fontsize=fontsize)
    axes[3].set_ylim(min_max_dict['Lockdown_duration'])
    
    for ax in axes:
        ax.set_box_aspect(1)
        
    plt.tight_layout()
    
    if name is not None:
        name_parts = str(name).rsplit('.', 1)
        scatter_save_name = f"{name_parts[0]}_densities.{name_parts[1]}" if len(name_parts) == 2 else f"{name}_densities"
        plt.savefig(scatter_save_name)
    plt.show()

    # ==========================================
    # PLOT 3: Separate Radial Plots
    # ==========================================
    print("Generating Separate Radial Plots...")

    param_cols_radial = ['R0V', 'R0Y', 'TV_years', 'TY_years', 'sigmaVY', 'sigmaYV', 'm', 'season_phase', 'c_pandemic', 'Lockdown_duration']

    short_labels = {
        'R0V': r"$R_{0,V}$",
        'R0Y': r"$R_{0,Y}$",
        'TV_years': r"$T_{V}$",
        'TY_years': r"$T_{Y}$",
        'sigmaVY': r"$\sigma_{VY}$",
        'sigmaYV': r"$\sigma_{YV}$",
        'm': r"$m$",
        'season_phase': r"$\phi$",
        'c_pandemic': r"$c_0$",
        'Lockdown_duration': r"$\tau$"
    }

    gray_df = plot_df[plot_df['Outcome'] == str1]
    green_df = plot_df[plot_df['Outcome'] == str3]
    blue_df = plot_df[plot_df['Outcome'] == str_blue]

    gray_mins = gray_df[param_cols_radial].min()
    gray_maxs = gray_df[param_cols_radial].max()

   # norm_gray = (gray_df[param_cols_radial] - gray_mins) / (gray_maxs - gray_mins)
    norm_green = (green_df[param_cols_radial] - gray_mins) / (gray_maxs - gray_mins)
    norm_blue = (blue_df[param_cols_radial] - gray_mins) / (gray_maxs - gray_mins)

    num_vars = len(param_cols_radial)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles_closed = angles + angles[:1]

    plot_configs = [
       # {'df_norm': norm_gray, 'color': 'gray', 'title': 'Baseline Coexistence (Gray)', 'suffix': '_radial_gray'},
        {'df_norm': norm_green, 'color': 'green', 'title': 'Vic Reemerges - All (Green)', 'suffix': '_radial_green'},
        {'df_norm': norm_blue, 'color': 'blue', 'title': 'Vic Reemerges - Favored (Blue)', 'suffix': '_radial_blue'}
    ]

    for config in plot_configs:
        fig3, ax = plt.subplots(figsize=(11, 11), subplot_kw=dict(polar=True))
        
        df_norm = config['df_norm']
        line_color = config['color']
        
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        for idx, row in df_norm.iterrows():
            values = row.tolist()
            values += values[:1]
            ax.plot(angles_closed, values, color=line_color, linewidth=0.8, alpha=0.02)

        ax.set_xticks(angles)
        ax.set_xticklabels([short_labels[col] for col in param_cols_radial], fontsize=18)
        ax.set_ylim(0, 1)
        
        ax.set_yticks([]) 
        ax.yaxis.grid(False) 
        
        ax.spines['polar'].set_color('dimgray')
        ax.spines['polar'].set_linewidth(1.5)

        plt.tight_layout()
        
        if name is not None:
            save_name = name.replace(".png", f"{config['suffix']}.png") if ".png" in name else name + config['suffix']
            plt.savefig(save_name)
            
        plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines


def plot_parameter_space_new(df, name=None):    
    """
    Takes the full DataFrame from the LHS sweep and plots the parameter spaces.
    Plots each sample as a scatter point coloured by outcome:
      - green: timely Victoria recovery (Oct 2021 – Apr 2022)
      - yellow: early Victoria recovery (before Oct 2021)
      - blue: late Victoria recovery (after Apr 2022)
    """
    coexistence_target = (df['simple_coexist'] == True) & (df['recovery_Y'] != -1) & (df['recovery_V'] != -1)
    Yextinction_Vrecovery = coexistence_target & (df['extinct_Y'] == True) & (df['extinct_V'] == False) & (df['recovery_Y'] != -100)

    # --- Define Outcome Conditions ---
    base_req = coexistence_target & (df['extinct_Y'] == True) & (df['recovery_Y'] != -100)

    cond_green = Yextinction_Vrecovery & (df['recovery_V'] > 1.5) & (df['recovery_V'] <= 2)
    cond_early = base_req & (df['extinct_V'] == False) & (df['recovery_V'] <= 1.5) & (df['recovery_V'] != -1)
    cond_late  = base_req & (df['extinct_V'] == False) & (df['recovery_V'] > 2)

    df_green = df[cond_green]
    df_early = df[cond_early]
    df_late  = df[cond_late]

    # Display labels
    display_labels = {
        'R0V': r"$R_{0,V}$",
        'R0Y': r"$R_{0,Y}$",
        'TV_years': r"$T_{V}$",
        'TY_years': r"$T_{Y}$",
        'sigmaVY': r"$\sigma_{VY}$",
        'sigmaYV': r"$\sigma_{YV}$",
        'm': r"$m$",
        'season_phase': r"$\phi$",
        'c_pandemic': r"$c_0$",
        'Lockdown_duration': r"$\tau$",
    }

    param_cols = list(display_labels.keys())

    # Axis limits from full dataframe
    min_max_dict = {col: (df[col].min(), df[col].max()) for col in param_cols}

    # ==========================================
    # PLOT
    # ==========================================
    print("Generating Categorized Parameter Space Subplots...")

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    color_green = '#2ca25f'
    color_early = '#fee090'
    color_late  = '#d73027' 

    plot_pairs = [
        ('R0Y', 'TY_years'),
        ('R0V', 'TV_years'),
        ('R0V', 'Lockdown_duration'),
        ('c_pandemic', 'Lockdown_duration')
    ]

    fontsize = 20

    for i, (x_col, y_col) in enumerate(plot_pairs):
        ax = axes[i]
        ax.text(0, 1, ['A', 'B', 'C', 'D'][i], transform=ax.transAxes,
                fontsize=25, fontname='Arial', fontweight='bold', va='bottom', ha='right')

        # Plot early and late first (background), then green on top
        ax.scatter(df_early[x_col], df_early[y_col], c=color_early, s=10, alpha=0.5, edgecolor='none')
        ax.scatter(df_late[x_col],  df_late[y_col],  c=color_late,  s=10, alpha=0.5, edgecolor='none')
        ax.scatter(df_green[x_col], df_green[y_col], c=color_green, s=10, alpha=1, edgecolor='none')

        # Formatting
        ax.set_xlabel(display_labels[x_col], fontsize=fontsize)
        plt.setp(ax.xaxis.get_majorticklabels(), fontsize=12)
        ax.set_xlim(min_max_dict[x_col])
        ax.set_ylabel(display_labels[y_col], fontsize=fontsize)
        plt.setp(ax.yaxis.get_majorticklabels(), fontsize=12)
        ax.set_ylim(min_max_dict[y_col])
        ax.set_box_aspect(1)

    # Legend
    legend_elements = [
        mlines.Line2D([], [], color=color_green, marker='o', linestyle='None',
                      markersize=8, alpha=1, label='Timely Victoria reemergence (October 2021 – March 2022)'),
        mlines.Line2D([], [], color=color_early, marker='o', linestyle='None',
                      markersize=8, alpha=0.5, label='Early Victoria reemergence (before October 2021)'),
        mlines.Line2D([], [], color=color_late,  marker='o', linestyle='None',
                      markersize=8, alpha=0.5, label='Late Victoria reemergence (after March 2022)')]

    fig.legend(handles=legend_elements, loc='upper center',
               bbox_to_anchor=(0.5, 1.15), ncol=3, fontsize=fontsize)

    plt.tight_layout()

    if name is not None:
        name_parts = str(name).rsplit('.', 1)
        scatter_save_name = (f"{name_parts[0]}_densities.{name_parts[1]}"
                             if len(name_parts) == 2 else f"{name}_densities")
        plt.savefig(scatter_save_name, bbox_inches='tight')

    plt.show()

In [ ]:
def count_and_plot(df, name=None):
    L=len(df)
    print('Number of successful ODE runs:', L)
    df=df[(df['recovery_Y'] !=-1) & (df['recovery_V'] != -1) & (df['simple_coexist'] == True) ]
    input_df = df
    print('Percentage of successful runs with pre-pandemic coexistence:', len(df)/L*100)
    df = df[(df['extinct_Y'] == True) & (df['extinct_V'] == False) & (df['recovery_Y'] != -100)]
    print('Percentage of successful runs with Y extinct and V surviving:', len(df)/L*100)
    print('Percentage of successful runs with early V recovery:', len(df[df['recovery_V'] <= 1.5])/L*100)
    target_runs = df[(df['recovery_V'] > 1.5) & (df['recovery_V'] <= 2)]
    print('Percentage of successful runs with appropiate V recovery:', len(target_runs)/L*100)
    print('Percentage of successful runs with late V recovery:', len(df[df['recovery_V'] > 2])/L*100)
    plot_parameter_space(input_df, name=name)
    return target_runs

In [ ]:
# Fig. 2 / S2 (parameter distributions) + summary statistics; returns the selected subset
target_runs = count_and_plot(all_res, 'both_outcomes.png')


In [ ]:
# Fig. 3: 2-D parameter-space projections -> evaluation_densities.png
plot_parameter_space_new(all_res, 'evaluation.png')


In [ ]:
target_runs.to_csv('target_runs.csv', index=False)


In [ ]:
filtered=target_runs[(target_runs['sigmaVY']>target_runs['sigmaYV']) & (target_runs['TV_years'] > target_runs['TY_years'])]
filtered.to_csv('filtered.csv', index=False)

## 4. Likelihood scoring and Northern-Hemisphere best fit (Fig. 4)

Negative-log-likelihood evaluation of the filtered samples against the FluNet data,
the best fit to the Northern-Hemisphere data, and its stochastic trajectory (`sto_nh.png`).

In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Make sure this is in your working directory or path
from simulator import model

# ==========================================
# 1. DATA PREPARATION
# ==========================================

obs_dfs_list = []

# Process each region's observed data
for region_df in [df_nh, df_tr, df_sh]:
    filtered_df = region_df[
        (region_df['ISO_WEEKSTARTDATE'] >= start_date) &
        (region_df['ISO_WEEKSTARTDATE'] < end_date)
    ].copy()
    
    for col in cols_to_fill:
        filtered_df[col] = filtered_df[col].fillna(0)
    
    # Group by week to aggregate counts
    filtered_weekly = filtered_df.groupby('ISO_WEEKSTARTDATE')[cols_to_fill].sum().reset_index()
    
    # Filter out weeks with 0 cases
    filtered_weekly = filtered_weekly[
        (filtered_weekly[BYAM] + filtered_weekly[BVIC] + filtered_weekly[BNOTDETERMINED]) > 0
    ].copy()
    
    obs_dfs_list.append(filtered_weekly)

# Convert date strings to datetime for reference
start_dt = pd.to_datetime(start_date)
end_dt = pd.to_datetime(end_date)

# Compute the simulator time (days) for start and end dates
t_ref_days = 15.75 * 365.25
t_ref_dt = datetime.datetime(2015, 10, 1, tzinfo=datetime.timezone.utc)

# ==========================================
# 2. CORE NLL COMPUTATION
# ==========================================

def compute_nll_for_params_all_regions(row, obs_dfs_list, verbose=False):
    """
    Compute negative log-likelihood for a single parameter set across all 3 regions.
    Uses different reporting rates before and after July 1, 2020 (pandemic).
    """
    R0V = row.R0V
    R0Y = row.R0Y
    TV = row.TV_years
    TY = row.TY_years
    sigmaVY = row.sigmaVY
    sigmaYV = row.sigmaYV
    m = row.m
    season_phase = row.season_phase
    c_pandemic = row.c_pandemic
    Lockdown_duration = row.Lockdown_duration
    
    # Run simulation once for all regions
    result = model(
        plot=False, stochastic=False, R0V=R0V, R0Y=R0Y, omegaV=1/(365.25*TV), omegaY=1/(365.25*TY),
        sigmaVY=sigmaVY, sigmaYV=sigmaYV, m=m, season_phase=season_phase,
        c_pandemic=c_pandemic, Lockdown_duration=Lockdown_duration
    )
    
    weekly_times = result['weekly_times']
    
    # Convert simulator times to ISO week dates
    def t_to_iso_date(t_days):
        """Convert simulation time (days) to ISO week date string."""
        days_from_ref = t_days - t_ref_days
        target_date = t_ref_dt + datetime.timedelta(days=days_from_ref)
        iso_cal = target_date.isocalendar()
        return f"{iso_cal[0]}-W{iso_cal[1]:02d}"
    
    sim_iso_dates = [t_to_iso_date(t) for t in weekly_times]
    
    # Pandemic cutoff date
    testing_cutoff = pd.to_datetime('2020-07-01')
    
    nll_total = 0.0
    rho_dict = {}
    predictions_dict = {}
    diagnostics = {'nll_by_region': {}}
    
    eps = 1e-10
    
    # Evaluate likelihood for each region
    for region_idx, obs_df in enumerate(obs_dfs_list):
        # Extract weekly predictions for this region
        weekly_V = result['weekly_infections_V'][region_idx]
        weekly_Y = result['weekly_infections_Y'][region_idx]
        
        # Match simulator weeks to observation weeks
        obs_iso_dates = obs_df['ISO_WEEKSTARTDATE'].dt.strftime('%Y-W%V').values
        
        muV_matched = []
        muY_matched = []
        
        for obs_iso in obs_iso_dates:
            if obs_iso in sim_iso_dates:
                idx = sim_iso_dates.index(obs_iso)
                muV_matched.append(weekly_V[idx])
                muY_matched.append(weekly_Y[idx])
            else:
                muV_matched.append(eps)
                muY_matched.append(eps)
        
        muV = np.array(muV_matched)
        muY = np.array(muY_matched)
        
        muV = np.maximum(muV, eps)
        muY = np.maximum(muY, eps)
        muT = np.maximum(muV + muY, eps)
        
        # Weekly typing probability from data
        obsV = obs_df[BVIC].values
        obsY = obs_df[BYAM].values
        obsN = obs_df[BNOTDETERMINED].values
        
        typed = obsV + obsY
        total = typed + obsN
        tau_t = np.where(total > 0, typed / total, 0.5)
        tau_t = np.clip(tau_t, eps, 1.0 - eps)
        
        # Separate pre- and post-pandemic data
        obs_dates = obs_df['ISO_WEEKSTARTDATE'].values
        pre_pandemic_mask = obs_dates < testing_cutoff
        post_pandemic_mask = obs_dates >= testing_cutoff
        
        # Profile out rho analytically for PRE-pandemic period
        sum_mu_total_pre = float(np.sum(muT[pre_pandemic_mask]))
        sum_obs_total_pre = float(np.sum(obsV[pre_pandemic_mask]) + np.sum(obsY[pre_pandemic_mask]) + np.sum(obsN[pre_pandemic_mask]))
        
        # Profile out rho analytically for POST-pandemic period
        sum_mu_total_post = float(np.sum(muT[post_pandemic_mask]))
        sum_obs_total_post = float(np.sum(obsV[post_pandemic_mask]) + np.sum(obsY[post_pandemic_mask]) + np.sum(obsN[post_pandemic_mask]))
        
        if sum_mu_total_pre <= 0 or sum_mu_total_post <= 0:
            return np.inf, None, None, {'error': f'sum_mu_total <= 0 for region {region_idx}', 'success': False}
        
        rho_pre = sum_obs_total_pre / sum_mu_total_pre if sum_obs_total_pre > 0 else eps
        rho_post = sum_obs_total_post / sum_mu_total_post if sum_obs_total_post > 0 else eps
        
        rho_dict[region_idx] = (rho_pre, rho_post)
        predictions_dict[region_idx] = {'muV': muV, 'muY': muY, 'obsV': obsV, 'obsY': obsY, 'obsN': obsN, 'tau_t': tau_t}
        
        # Compute negative log-likelihood for PRE-pandemic period
        ll_core_pre = (
            np.sum(obsV[pre_pandemic_mask] * (np.log(tau_t[pre_pandemic_mask]) + np.log(muV[pre_pandemic_mask])))
            + np.sum(obsY[pre_pandemic_mask] * (np.log(tau_t[pre_pandemic_mask]) + np.log(muY[pre_pandemic_mask])))
            + np.sum(obsN[pre_pandemic_mask] * (np.log(1.0 - tau_t[pre_pandemic_mask]) + np.log(muT[pre_pandemic_mask])))
        )
        
        nll_pre = sum_obs_total_pre * np.log(sum_mu_total_pre) - ll_core_pre
        
        # Compute negative log-likelihood for POST-pandemic period
        ll_core_post = (
            np.sum(obsV[post_pandemic_mask] * (np.log(tau_t[post_pandemic_mask]) + np.log(muV[post_pandemic_mask])))
            + np.sum(obsY[post_pandemic_mask] * (np.log(tau_t[post_pandemic_mask]) + np.log(muY[post_pandemic_mask])))
            + np.sum(obsN[post_pandemic_mask] * (np.log(1.0 - tau_t[post_pandemic_mask]) + np.log(muT[post_pandemic_mask])))
        )
        
        nll_post = sum_obs_total_post * np.log(sum_mu_total_post) - ll_core_post
        nll_region = nll_pre + nll_post
        
        # Store the individual region's NLL
        diagnostics['nll_by_region'][region_idx] = nll_region
        
        nll_total += nll_region
    
    diagnostics['success'] = True
    
    if verbose:
        print(f"Total NLL (all regions): {nll_total:.2f}")
    
    return float(nll_total), rho_dict, predictions_dict, diagnostics

# ==========================================
# 3. EVALUATION FUNCTIONS
# ==========================================

def evaluate_parameter_sets_likelihood_all(param_df, obs_dfs_list, verbose=True):
    """
    Evaluate likelihood (NLL) for all pre-selected parameter sets across all regions.
    """
    results = []
    
    for idx, row in param_df.iterrows():
        if verbose and (idx % max(1, len(param_df) // 20) == 0):
            print(f"Evaluating parameter set {idx+1}/{len(param_df)}...")
        
        nll, rho_dict, predictions_dict, diagnostics = compute_nll_for_params_all_regions(
            row, obs_dfs_list, verbose=False)
        
        result_row = {
            'param_idx': idx,
            'nll': nll,
            'success': diagnostics.get('success', False),
        }
        
        if rho_dict is not None:
            for r_idx, r_name in enumerate(['NH', 'Tr', 'SH']):
                rho_pre, rho_post = rho_dict[r_idx]
                result_row[f'rho_{r_name}_pre'] = rho_pre
                result_row[f'rho_{r_name}_post'] = rho_post
        
        # Add parameter values to results
        for col in param_df.columns:
            result_row[f'p_{col}'] = row[col]
        
        results.append(result_row)
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('nll').reset_index(drop=True)
    
    return results_df


def find_and_plot_best_fit_for_region(param_df, obs_dfs_list, region_to_plot, verbose=True):
    """
    Evaluates parameter sets to find the minimum NLL for a SPECIFIC region, 
    and passes the best row to the plotter.
    
    Parameters:
    - param_df: DataFrame containing the parameter sets to evaluate
    - obs_dfs_list: list of 3 observation DataFrames
    - region_to_plot: 'nh', 'tr', or 'sh'
    """
    region_map = {'nh': 0, 'tr': 1, 'sh': 2}
    
    if region_to_plot.lower() not in region_map:
        raise ValueError("region_to_plot must be 'nh', 'tr', or 'sh'")
        
    target_idx = region_map[region_to_plot.lower()]
    
    best_nll = np.inf
    best_row = None
    
    if verbose:
        print(f"Searching for best parameter set for {region_to_plot.upper()}...")
        
    for idx, row in param_df.iterrows():
        nll_total, rho_dict, preds, diag = compute_nll_for_params_all_regions(row, obs_dfs_list, verbose=False)
        
        if diag.get('success', False):
            # Extract the NLL for just our target region
            region_nll = diag['nll_by_region'][target_idx] 
            
            # Keep track of the lowest regional NLL found
            if region_nll < best_nll:
                best_nll = region_nll
                best_row = row.copy()
                
    if best_row is None:
        print(f"Error: Could not find a valid fit for {region_to_plot.upper()}.")
        return None
        
    if verbose:
        print(f"Found Best Fit for {region_to_plot.upper()}! (Regional NLL: {best_nll:.2f})")
        
    # Plot only the relevant region using the best parameters found for it
    plot_best_fit(best_row, obs_dfs_list, region_to_plot=region_to_plot)
    
    return best_row

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================

def plot_best_fit(best_params_row, obs_dfs_list, region_to_plot='all'):
    """
    Plot the best-fitting parameter set against observed data.
    
    Parameters:
    - best_params_row: row containing best parameters (Pandas Series)
    - obs_dfs_list: list of 3 observation DataFrames
    - region_to_plot: 'all', 'nh', 'tr', or 'sh'
    """
    nll, rho_dict, predictions_dict, diag = compute_nll_for_params_all_regions(
        best_params_row, obs_dfs_list, verbose=False)
    
    if predictions_dict is None:
        print("Error: could not compute predictions for best parameter set")
        return
    
    testing_cutoff = pd.to_datetime('2020-07-01')
    
    # Mapping inputs to region indices
    region_map = {'nh': 0, 'tr': 1, 'sh': 2}
    region_names = ["Northern Hemisphere", "Tropics", "Southern Hemisphere"]
    
    # Configure subplots based on region selection
    if region_to_plot.lower() == 'all':
        fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
        indices_to_plot = [0, 1, 2]
    else:
        if region_to_plot.lower() not in region_map:
            raise ValueError("region_to_plot must be 'all', 'nh', 'tr', or 'sh'")
        
        fig, ax = plt.subplots(1, 1, figsize=(15, 5))
        axes = [ax] 
        indices_to_plot = [region_map[region_to_plot.lower()]]

    # Plot logic
    for ax, r_idx in zip(axes, indices_to_plot):
        obs_df = obs_dfs_list[r_idx]
        preds = predictions_dict[r_idx]
        rho_pre, rho_post = rho_dict[r_idx]
        
        muV = preds['muV']
        muY = preds['muY']
        obsV = preds['obsV']
        obsY = preds['obsY']
        obsN = preds['obsN']
        tau_t = preds['tau_t']
        
        x_dates = obs_df['ISO_WEEKSTARTDATE']
        pre_mask = x_dates < testing_cutoff
        
        muV_scaled = np.where(pre_mask, rho_pre * muV, rho_post * muV)
        muY_scaled = np.where(pre_mask, rho_pre * muY, rho_post * muY)
        
        obsV_pred = tau_t * muV_scaled
        obsY_pred = tau_t * muY_scaled
        obsN_pred = (1.0 - tau_t) * (muV_scaled + muY_scaled)
        
        ax.plot(x_dates, np.sqrt(obsV), 'o', label='Obs Victoria', alpha=0.6, markersize=4)
        ax.plot(x_dates, np.sqrt(obsY), 's', label='Obs Yamagata', alpha=0.6, markersize=4)
        ax.plot(x_dates, np.sqrt(obsN), '^', label='Obs Untyped', alpha=0.6, markersize=4)
        
        ax.plot(x_dates, np.sqrt(obsV_pred), '--', label='Pred Victoria', linewidth=2, alpha=0.8, color='C0')
        ax.plot(x_dates, np.sqrt(obsY_pred), '--', label='Pred Yamagata', linewidth=2, alpha=0.8, color='C1')
        ax.plot(x_dates, np.sqrt(obsN_pred), '--', label='Pred Untyped', linewidth=2, alpha=0.8, color='C2')
        
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
        ax.set_ylabel('Weekly Reports (sqrt scale)', fontsize=11)
        ax.set_title(f'{region_names[r_idx]} (Ï_pre={rho_pre*100:.3f}%, Ï_post={rho_post*100:.3f}%)', 
                     fontsize=12, fontweight='bold')
        ax.legend(ncol=2, fontsize=9)
        ax.grid(True, alpha=0.3)
    
    # Format the x-axis for the bottom-most plot
    axes[-1].set_xlabel('ISO Week Start Date', fontsize=11)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[-1].set_xlim(x_dates.min(), x_dates.max())
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Suptitle identifies the global NLL score for this parameter set
    plt.suptitle(f'Best-Fit Parameter Set (Global NLL={nll:.2f})', fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()



################GLOBAL BEST

def plot_best_fit_all_regions(best_params_row, obs_dfs_list):
    """
    Plot the best-fitting parameter set against observed data for all 3 regions.
    """
    nll, rho_dict, predictions_dict, diag = compute_nll_for_params_all_regions(
        best_params_row, obs_dfs_list, verbose=False)
    
    if predictions_dict is None:
        print("Error: could not compute predictions for best parameter set")
        return
    
    testing_cutoff = pd.to_datetime('2020-07-01')
    region_names = ["Northern Hemisphere", "Tropics", "Southern Hemisphere"]
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
    
    for r_idx, (ax, obs_df) in enumerate(zip(axes, obs_dfs_list)):
        preds = predictions_dict[r_idx]
        rho_pre, rho_post = rho_dict[r_idx]
        
        muV = preds['muV']
        muY = preds['muY']
        obsV = preds['obsV']
        obsY = preds['obsY']
        obsN = preds['obsN']
        tau_t = preds['tau_t']
        
        x_dates = obs_df['ISO_WEEKSTARTDATE']
        pre_mask = x_dates < testing_cutoff
        
        muV_scaled = np.where(pre_mask, rho_pre * muV, rho_post * muV)
        muY_scaled = np.where(pre_mask, rho_pre * muY, rho_post * muY)
        
        obsV_pred = tau_t * muV_scaled
        obsY_pred = tau_t * muY_scaled
        obsN_pred = (1.0 - tau_t) * (muV_scaled + muY_scaled)
        
        ax.plot(x_dates, np.sqrt(obsV), 'o', label='Obs Victoria', alpha=0.6, markersize=4)
        ax.plot(x_dates, np.sqrt(obsY), 's', label='Obs Yamagata', alpha=0.6, markersize=4)
        ax.plot(x_dates, np.sqrt(obsN), '^', label='Obs Untyped', alpha=0.6, markersize=4)
        
        ax.plot(x_dates, np.sqrt(obsV_pred), '--', label='Pred Victoria', linewidth=2, alpha=0.8, color='C0')
        ax.plot(x_dates, np.sqrt(obsY_pred), '--', label='Pred Yamagata', linewidth=2, alpha=0.8, color='C1')
        ax.plot(x_dates, np.sqrt(obsN_pred), '--', label='Pred Untyped', linewidth=2, alpha=0.8, color='C2')
        
        #ax.axvline(testing_cutoff, color='red', linestyle='--', alpha=0.7, label='Testing Cutoff ')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: f'{int(val**2):,}'))
        ax.set_ylabel('Weekly Reports (sqrt scale)', fontsize=11)
        ax.set_title(f'{region_names[r_idx]} (Ï_pre={rho_pre*100:.3f}%, Ï_post={rho_post*100:.3f}%)', 
                     fontsize=12, fontweight='bold')
        ax.legend(ncol=2, fontsize=9)
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('ISO Week Start Date', fontsize=11)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[-1].set_xlim(x_dates.min(), x_dates.max())
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.suptitle(f'Best-Fit Parameter Set (Total NLL={nll:.2f})', fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()


In [ ]:
find_and_plot_best_fit_for_region(filtered, obs_dfs_list, 'nh', verbose=True)

In [ ]:
# Stochastic trajectory of the NH best-fit sample -> sto_nh.png (Fig. 4).
# These are the published NH best-fit parameters (as found by the cell above).
# NOTE: nsims/nsims2 = 100000 reproduces the paper but is slow; lower them for a quick check.
from simulator import model
R0V             =                     3.868179
R0Y                =                  2.303436
TV_years            =                 9.881269
TY_years             =                5.517838
sigmaVY               =               0.154662
sigmaYV               =              0.01492
m                     =             0.007065
season_phase          =            15.503095
c_pandemic            =           0.300247
Lockdown_duration     =          569.090759
sims=100000
res=model(stochastic=True, nsims=sims, tspan=27.75*365.25, Tpan=20.25/27.75, stochastic2=True,nsims2=sims, name='sto_nh.png', R0V=R0V, R0Y=R0Y, omegaV=1/(365.25*TV_years), omegaY=1/(365.25*TY_years), sigmaVY=sigmaVY, sigmaYV=sigmaYV, m=m, season_phase=season_phase, c_pandemic=c_pandemic, Lockdown_duration=Lockdown_duration)

## 5. Reintroduction analysis (Fig. 5) and supplementary scatter plots

Yamagata's effective reproduction number at four hypothetical reintroduction dates
(`Yam.png`, Fig. 5), plus the likelihood-vs-metric scatter plots.

**Runtime note:** the loops below run the deterministic model once per filtered sample
(559 of them) several times over, so they can take a while.

In [ ]:
import numpy as np

def add_region_nll_column(param_df, obs_dfs_list, region='nh'):
    """
    Computes the NLL for a specific region (or overall) for every row in param_df 
    and adds it as a new column (e.g., 'NLL_nh' or 'NLL_all').
    
    Parameters:
    - param_df: DataFrame containing the parameter sets
    - obs_dfs_list: list of 3 observation DataFrames
    - region: 'nh', 'tr', 'sh', or 'all' (defaults to 'nh')
    
    Returns:
    - A copy of param_df with the new NLL column added.
    """
    region = region.lower()
    region_map = {'nh': 0, 'tr': 1, 'sh': 2}
    
    # Check if the region is valid (either in our map or exactly 'all')
    if region not in region_map and region != 'all':
        raise ValueError("region must be 'nh', 'tr', 'sh', or 'all'")
        
    # Get the index if it's a specific region, otherwise this will be None
    target_idx = region_map.get(region)
    
    nll_values = []
    
    for idx, row in param_df.iterrows():
        # Unpack the results
        nll_total, rho_dict, preds, diag = compute_nll_for_params_all_regions(row, obs_dfs_list, verbose=False)
        
        # Check if the computation was successful
        if diag.get('success', False):
            if region == 'all':
                # Grab the overall NLL returned by the compute function
                nll_values.append(nll_total)
            else:
                # Grab the specific regional NLL from the diagnostics dictionary
                region_nll = diag['nll_by_region'][target_idx]
                nll_values.append(region_nll)
        else:
            # If the fit failed, append NaN
            nll_values.append(np.nan) 
            
    # Create a copy to avoid Pandas SettingWithCopy warnings
    result_df = param_df.copy()
    
    # Add the collected values as a new column
    col_name = f'NLL_{region}'
    result_df[col_name] = nll_values
    
    return result_df

In [ ]:
filtered=pd.read_csv('filtered.csv')
from simulator import model
RY = []
min_list = []
nll_list = []
R_dif=[]
filtered_nll_nh = add_region_nll_column(filtered, obs_dfs_list, region='nh')
for _, row in filtered_nll_nh.iterrows():
    res = model(R0V=row['R0V'],R0Y=row['R0Y'],omegaV=1/(365.25 * row['TV_years']),omegaY=1/(365.25 * row['TY_years']),sigmaVY=row['sigmaVY'], sigmaYV=row['sigmaYV'],m=row['m'],
     season_phase=row['season_phase'],c_pandemic=row['c_pandemic'],Lockdown_duration=row['Lockdown_duration'],plot=False,stochastic=False)
    RY.append(res['R_eff_Y_peak_2026_27'])
    min_list.append(res['min_prevalence_V'])
    nll_list.append(row['NLL_nh'])
    R_dif.append(row['R0V']-row['R0Y'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Force both inputs to be flat 1D arrays
min_list = np.asarray(min_list).flatten()
nll_list = np.asarray(nll_list).flatten()

# Find the index of the best fit (minimum NLL)
best_idx = np.argmin(nll_list)
best_y = min_list[best_idx]
best_x = nll_list[best_idx]

fig, ax = plt.subplots(figsize=(8, 6))

ax.set_yscale('log')  # Set y-axis to logarithmic scale

# Plot all points, coloring them by their NLL value
scatter = ax.scatter(
    nll_list, min_list,
    #c=nll_list, 
    #cmap='viridis', 
    alpha=0.6, 
    edgecolors='w', color='darkblue',
    linewidth=0.5,
    label=f'{len(nll_list)} biologically-filtered samples'
)

# Highlight the absolute best point (lowest NLL) in red
ax.scatter(
    best_x, 
    best_y, 
    color='red', 
    edgecolor='black', 
    s=150, 
    zorder=5, 
    marker='*', 
    label=f'Best fit (minimum Victoria prevalence: {best_y:.0f})'
)

# Formatting
ax.set_xlabel('Negative Log-Likelihood', fontsize=14)
ax.set_ylabel('Minimum Victoria prevalence', fontsize=14)

# Add a colorbar
#cbar = plt.colorbar(scatter)
#cbar.set_label('NLL')

plt.legend(fontsize=14)
plt.tight_layout()
plt.grid()
plt.savefig('Vic_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Force both inputs to be flat 1D arrays
RY = np.asarray(RY).flatten()
nll_list = np.asarray(nll_list).flatten()

# Find the index of the best fit (minimum NLL)
best_idx = np.argmin(nll_list)
best_y = RY[best_idx]
best_x = nll_list[best_idx]

fig, ax = plt.subplots(figsize=(8, 6))


# Plot all points, coloring them by their NLL value
scatter = ax.scatter(
    nll_list, RY,
   # c=nll_list, 
    #cmap='viridis', 
    alpha=0.6, 
    edgecolors='w', color='darkblue',
    linewidth=0.5,
    label=f'{len(nll_list)} biologically-filtered samples'
)

# Highlight the absolute best point (lowest NLL) in red
ax.scatter(
    best_x, 
    best_y, 
    color='red', 
    edgecolor='black', 
    s=150, 
    zorder=5, 
    marker='*', 
    label=f'Best fit (effective R-number: {best_y:.2f})'
)

# Formatting
ax.set_xlabel('Negative Log-Likelihood', fontsize=14)
ax.set_ylabel('Peak 2026-27 effective Yamagata R-number', fontsize=14)

# Add a colorbar
#cbar = plt.colorbar(scatter)
#cbar.set_label('NLL')

plt.legend(fontsize=14)
plt.tight_layout()
plt.grid()
plt.savefig('Yam_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Force both inputs to be flat 1D arrays
R_dif = np.asarray(R_dif).flatten()
nll_list = np.asarray(nll_list).flatten()

# Find the index of the best fit (minimum NLL)
best_idx = np.argmin(nll_list)
best_y = R_dif[best_idx]
best_x = nll_list[best_idx]

fig, ax = plt.subplots(figsize=(8, 6))


# Plot all points, coloring them by their NLL value
scatter = ax.scatter(
    nll_list, R_dif,
   # c=nll_list, 
    #cmap='viridis', 
    alpha=0.6, 
    edgecolors='w', color='darkblue',
    linewidth=0.5,
    label=f'{len(nll_list)} biologically-filtered samples'
)

# Highlight the absolute best point (lowest NLL) in red
ax.scatter(
    best_x, 
    best_y, 
    color='red', 
    edgecolor='black', 
    s=150, 
    zorder=5, 
    marker='*', 
    label=f'Best fit (' +r'$R_{0,V} - R_{0,Y}$' + f': {best_y:.2f})'
)

# Formatting
ax.set_xlabel('Negative Log-Likelihood', fontsize=14)
ax.set_ylabel(r'$R_{0,V} - R_{0,Y}$', fontsize=14)

# Add a colorbar
#cbar = plt.colorbar(scatter)
#cbar.set_label('NLL')

plt.legend(fontsize=14)
plt.tight_layout()
plt.grid()
plt.savefig('Yam_dif.png', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import pandas as pd
from simulator import model
R1, R2, R3 , R4= [], [], [], []
filtered=pd.read_csv('filtered.csv')
for _, row in filtered.iterrows():
    res = model(tspan=27.75*365.25, Tpan=20.25/27.75, R0V=row['R0V'], R0Y=row['R0Y'],omegaV=1/(365.25 * row['TV_years']),omegaY=1/(365.25 * row['TY_years']),sigmaVY=row['sigmaVY'], sigmaYV=row['sigmaYV'],m=row['m'],
     season_phase=row['season_phase'],c_pandemic=row['c_pandemic'],Lockdown_duration=row['Lockdown_duration'],plot=False,stochastic2=True, nsims2=0)
    R_val=res['R_values']
    R1.append(R_val[0])
    R2.append(R_val[1])
    R3.append(R_val[2])
    R4.append(R_val[3])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# 1. Define your custom labels here
labels = [
    "Reintroduction on Oct 1st, 2026",
    "Reintroduction on Jan 1st, 2027",
    "Reintroduction on April 1st, 2027",
    "Reintroduction on July 1st, 2027",
]

# 2. Find the global min and max across all 4 datasets
all_mins = [min(R1), min(R2), min(R3), min(R4)]
all_maxs = [max(R1), max(R2), max(R3), max(R4)]
global_min = min(all_mins)
global_max = max(all_maxs)
R_values=[1.56,1.78,1.64,1.46]
p=['44%','52%','45%','39%']
# 3. Create the figure
fig, axes = plt.subplots(nrows=4, ncols=1, sharex=True, sharey=False, figsize=(8, 11))
max_kde=0
for i in range(4):
    R = locals()[f"R{i+1}"]
    kde = gaussian_kde(R)
    
    x = np.linspace(global_min, global_max, 500)
    ax = axes[i]
    ax.text(0, 1, ['A', 'B', 'C', 'D'][i], transform=ax.transAxes,
                        fontsize=20, fontname='Arial', fontweight='bold', va='bottom', ha='right')
    kde_values = kde(x)
    max_kde = max(max_kde, kde_values.max())
    # Plot the KDE
    ax.plot(x, kde_values, color="black")
    ax.fill_between(x, kde_values, color="black", alpha=0.3)
    # Best fit results
    r=R_values[i]
    ax.plot(r, 0.1, '*', markersize=10, color="red")
    anno='Prob(outbreak > 100 cases) = ' + p[i]
    ax.annotate(anno, xy=(r-0.15, 0.5), fontsize=10, color="red")
    ax.set_title(labels[i], fontsize=14, fontweight='bold')
# 4. Global X-axis adjustments
plt.xlim(global_min, global_max)
plt.xticks([global_min,1, 1.25, 1.5, 1.75, global_max], ['{:.2f}'.format(x) for x in [global_min,1, 1.25, 1.5, 1.75, global_max]])

axes[-1].set_xlabel(
    "Yamagata's effective reproduction number (in NH)", fontsize=14)
for i in range(4):
    ax=axes[i]
    ax.set_ylim(0, 1.05*max_kde)
    ax.set_yticks([1,2,3])
    ax.set_yticklabels(['', '', ''])
    ax.set_ylabel('density', fontsize=14)
    ax.axvline(x=1, color='gray', linestyle='-', linewidth=1, alpha=0.3)

# 5. ADJUST SPACE BETWEEN PLOTS
# hspace controls the vertical padding between subplots.
# Increase this value (e.g., 0.6, 0.8) if you need even more room for labels.
plt.tight_layout()
plt.savefig('Yam.png', bbox_inches='tight')  # bbox_inches ensures labels aren't cut off
plt.show()